In [ ]:
# Common imports + DES (Modelica) translator pieces.
import json
import shutil

import pandas as pd
from geojson_modelica_translator.modelica.GMT_Lib.DHC.DHC_5G_WH_GHX_HPTrio_VariableDist import (
    DHC5GWasteHeatGHXwithHPTrioVariableDist,
)
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)
from urbanopt_des.urbanopt_analysis import (
    URBANoptAnalysis as _UOA,
)  # local alias for clarity
from urbanopt_des.urbanopt_geojson import DESGeoJSON as URBANoptGeoJSON

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
paths = setup_notebook_paths()
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

# update weather flag (optional)
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"
weather_override = (
    (new_weather_epw, new_weather_climate_zone) if update_weather_location else None
)

### Activity 4: DES/TEN

In [ ]:
# Copy activity_2/coincident to activity_4/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_2") / "coincident"
activity_4_dir = paths.activity_dir("activity_4")
dest_dir = activity_4_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_4/coincident
uo_coincident = UO(activity_4_dir, "coincident", template_dir=template_data_dir)

In [ ]:
# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Now create the 4G system params from the UO baseline.
uo_coincident.des_params(scenario_path, feature_path, sys_param_path)

In [ ]:
# Fix values in the sys_params.json file for the 4G system if needed.
# Example overrides:
#   "central_cooling_plant_parameters": {
#     "heat_flow_nominal": 8_000_000,   # 8 MW, was 7999 W
#     "cooling_tower_fan_power_nominal": 50_000,
#     "mass_chw_flow_nominal": 85,      # ~Σ building chw flows
#     "chiller_water_flow_minimum": 20, # ~0.25 × nominal
#     "mass_cw_flow_nominal": 100,      # ~1.15 × chw nominal — prevents freeze
#   }
#
# Build the 4G Modelica model from sys_params + feature file.
uo_coincident.des_create(sys_param_path, feature_path, "four_g")

In [ ]:
# 4G run via the gmt-om-runner. Gated so it does not run accidentally.
RUN_4G = False
if RUN_4G:
    activity_4_dir = paths.activity_dir("activity_4")
    des_scenario_name = "four_g"
    des_analysis_4g_dir = activity_4_dir / des_scenario_name

    print(f"{des_scenario_name}.Districts.district")
    print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
    print(f"run_path={des_analysis_4g_dir}")

    results_path = (
        des_analysis_4g_dir
        / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
    )
    if results_path.exists():
        print("Results already exist, skipping model run.")
    else:
        mr = ModelicaRunner()
        _, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.DistrictEnergySystem",
            file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
            run_path=des_analysis_4g_dir,
            # March 1 → March 14
            start_time=5097600,
            stop_time=6307200,
        )

In [ ]:
# 5G model from the uo cli workflow — does NOT WORK YET; left as a placeholder.
# sys_param_path_5g = uo_coincident.project_path / "sys_params_5g.json"
# uo_coincident.des_create(sys_param_path_5g, feature_path, "five_g")


### 5G: individual loads (will take a long time to run)

In [ ]:
# # Build a 5G scenario from the GMT template using the URBANopt baseline inputs.
# # WARNING: this will REMOVE the target 5G project directory if it exists.
# activity_4_dir = paths.activity_dir("activity_4")
# new_geojson_file = feature_path
# uo_analysis_baseline_results = uo_coincident.project_path / "run" / "baseline_scenario"
# des_analysis_agg_dir = activity_4_dir

# if not uo_analysis_baseline_results.exists():
#     raise FileNotFoundError(
#         f"Baseline scenario results not found: {uo_analysis_baseline_results}"
#     )

# # Important: use the repo root as relative_path so modelica:// paths resolve
# # to esbe_2026/... and not esbe/esbe_2026/...
# relative_path_root = workdir

# system_parameter_file = (
#     des_analysis_agg_dir / f"{new_geojson_file.stem}_system_params_5g.json"
# )
# system_parameter = SystemParameters()
# system_parameter.csv_to_sys_param(
#     model_type="time_series",
#     sys_param_filename=system_parameter_file,
#     scenario_dir=uo_analysis_baseline_results,
#     feature_file=new_geojson_file,
#     district_type="5G",
#     relative_path=relative_path_root,
#     modelica_load_filename="modelica.mos",
# )

# model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)

# des_scenario_name = "five_g"
# des_analysis_baseline_dir = des_analysis_agg_dir / des_scenario_name
# print(f"DES analysis dir for this scenario is {des_analysis_baseline_dir}")

# if des_analysis_baseline_dir.exists():
#     shutil.rmtree(des_analysis_baseline_dir)

# model.build_from_template(des_analysis_agg_dir, des_scenario_name)

# # Save the scenario name for downstream post-processing
# (des_analysis_baseline_dir / "analysis_name.txt").write_text(des_scenario_name)
# print("Created 5G model")

In [ ]:
# # Run the generated 5G template model.
# RUN_5G = False
# if RUN_5G:
#     des_scenario_name = "five_g"
#     activity_4_dir = paths.activity_dir("activity_4")
#     des_analysis_baseline_dir = activity_4_dir / des_scenario_name

#     if not (des_analysis_baseline_dir / "package.mo").exists():
#         raise FileNotFoundError(
#             f"Expected generated 5G package not found: {des_analysis_baseline_dir / 'package.mo'}"
#         )

#     print(f"{des_scenario_name}.Districts.district")
#     print(f"file_to_load={des_analysis_baseline_dir / 'package.mo'}")
#     print(f"run_path={des_analysis_baseline_dir}")

#     results_path = (
#         des_analysis_baseline_dir / f"{des_scenario_name}.Districts.district_results"
#     )
#     if results_path.exists():
#         print("Results already exist, skipping model run.")
#     else:
#         mr = ModelicaRunner()
#         success, results_path = mr.run_in_docker(
#             "compile_and_run",
#             f"{des_scenario_name}.Districts.district",
#             file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
#             run_path=des_analysis_baseline_dir,
#             step_size=900,  # 15 min step size
#             start_time=5097600,  # March 1
#             stop_time=6307200,  # March 14
#         )
#         print(f"Run success: {success}")
#     print(f"5G results path: {results_path}")

### 5G: Run with aggregated loads (much faster)

In [ ]:
# 5G aggregated model — aggregate every URBANopt building load into a single
# "all_buildings" feature, then build / run the GMT 5G template against it.

uo_analysis_baseline_dir = uo_coincident.project_path
uo_analysis_baseline_scenario_name = "baseline_scenario"
uo_analysis_baseline_results = (
    uo_analysis_baseline_dir / "run" / uo_analysis_baseline_scenario_name
)
project_geojson_filename = feature_path
des_analysis_agg_dir = activity_4_dir
relative_path_root = workdir

uo_analysis = _UOA(project_geojson_filename, des_analysis_agg_dir)
uo_analysis.add_urbanopt_results(
    uo_analysis_baseline_dir, uo_analysis_baseline_scenario_name
)
uo_analysis.urbanopt.process_load_results(uo_analysis.geojson.get_building_ids())
uo_analysis.urbanopt.save_dataframes()
uo_analysis.urbanopt.display_name = "Non-Connected"

# 1. Aggregated DES project to roll up loads.
uo_analysis.urbanopt.create_abstract_run(
    "all_buildings", uo_analysis.urbanopt.data_loads
)

# 2. Copy geojson file into the aggregated DES directory.
new_geojson_file = des_analysis_agg_dir / project_geojson_filename.name
shutil.copy(project_geojson_filename, des_analysis_agg_dir)

# 3. Update the GeoJSON to a single aggregated feature.
agg_geojson = URBANoptGeoJSON(new_geojson_file)
gdf_json = agg_geojson.create_aggregated_representation(["all"])
new_geojson_file.write_text(json.dumps(gdf_json, indent=2))

# 4. Update the scenario CSV so it has a single "All Buildings" row.
new_scenario_file = des_analysis_agg_dir / scenario_path.name
shutil.copy(scenario_path, des_analysis_agg_dir)
scenario_df = pd.read_csv(new_scenario_file)
scenario_df = scenario_df.head(1)
scenario_df["Feature Id"] = "all_buildings"
scenario_df["Feature Name"] = "All Buildings"
scenario_df.to_csv(new_scenario_file, index=False)

# 5. Generate system parameters for the aggregated rep.
system_parameter_file = (
    des_analysis_agg_dir / f"{new_geojson_file.stem}_system_params.json"
)
system_parameter = SystemParameters()
system_parameter.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=system_parameter_file,
    scenario_dir=uo_analysis_baseline_results,
    feature_file=new_geojson_file,
    district_type="5G",
    relative_path=relative_path_root,
    modelica_load_filename="modelica.mos",
)

model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)
des_scenario_name = "five_g_agg"
des_analysis_baseline_dir = des_analysis_agg_dir / des_scenario_name
if des_analysis_baseline_dir.exists():
    shutil.rmtree(des_analysis_baseline_dir)

model.build_from_template(des_analysis_agg_dir, des_scenario_name)
(des_analysis_baseline_dir / "analysis_name.txt").write_text(des_scenario_name)
print("Created 5G aggregated model")

In [ ]:
# Run the generated 5G aggregated template model. Full-year ≈ 100 min.
full_year_run = True
activity_4_dir = paths.activity_dir("activity_4")
des_scenario_name = "five_g_agg"
des_analysis_baseline_dir = activity_4_dir / des_scenario_name

if not (des_analysis_baseline_dir / "package.mo").exists():
    raise FileNotFoundError(
        f"Expected generated 5G package not found: {des_analysis_baseline_dir / 'package.mo'}"
    )

print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_baseline_dir / 'package.mo'}")
print(f"run_path={des_analysis_baseline_dir}")

results_path = (
    des_analysis_baseline_dir / f"{des_scenario_name}.Districts.district_results"
)
if results_path.exists():
    print("Results already exist, skipping model run.")
else:
    mr = ModelicaRunner()
    if full_year_run:
        success, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.district",
            file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
            run_path=des_analysis_baseline_dir,
            step_size=300,  # 5-min step
            stop_time=31_536_000,  # full year
        )
    else:
        success, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.district",
            file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
            run_path=des_analysis_baseline_dir,
            step_size=300,
            start_time=5_097_600,  # March 1
            stop_time=6_307_200,  # March 14
        )
        print(f"Run success: {success}")
print(f"5G results path: {results_path}")